# Notebook 06 — Manifold Visualisation
## Latent Behavioral State Machines (LBSM): Manifold Geometry of Adaptive Agent Telemetry

---

**Chain of evidence so far:**

| Notebook | Method | Verdict |
|----------|--------|---------|
| NB01 | PCA + LDA | 3/5 — moderate linear structure |
| NB02 | UMAP + t-SNE | 5/6 — strong nonlinear manifold geometry |
| NB03 | HMM | 5/6 — unsupervised regime recovery; unstable recall 100% |
| NB04 | Composite anomaly scorer | 5/6 — F1=0.872, latency=0.29 steps |
| NB05 | Q-learning over latent behavioral space | Reward ↑, unstable dwell ↓, transition entropy ↓ |
| **NB06** | **3-D manifold overlay of RL trajectories** | **Does training visibly move agents through the NB02 manifold?** |

**Central question:**
> *When the full per-step trajectory of a training run is projected into the exact NB02 UMAP
> manifold, does it trace an interpretable path — visibly retreating from the unstable region
> as training progresses, and does the action taken at each point correspond to the direction
> of that movement?*

NB05 answered this indirectly, via aggregate per-episode statistics (dwell fractions, episode
UMAP centroids, transition entropy) — never the actual per-step path through the embedding.
This notebook closes that gap directly: it reproduces a training run with full per-step
trajectory capture, projects every visited point into the NB02 manifold via a freshly-fit
UMAP reducer (validated against NB02's saved embedding), and overlays it colour-coded by
training phase (early/mid/late) and by the action taken at each step.

**Data provenance note:** NB05 persisted only the final learned Q-tables
(`data/raw/nb05/q_tables.npy`) and per-episode aggregate statistics — not the raw per-step
trajectory across a full training run (`BehavioralEnv.trajectory` only ever holds the *last*
episode; nothing captured it across all 120). This notebook regenerates that trajectory by
retraining a small subset of agents with identical seeds/hyperparameters
(`configs/rl.yaml`) and a new opt-in `collect_trajectories=True` flag added to
`QLearningAgent.train()` for exactly this purpose. The retrained Q-tables are validated
against `q_tables.npy` below to confirm this reproduces NB05's actual result, not an
approximation of it.

**Reference** — *"Latent Behavioral State Machines: Manifold Geometry of Adaptive Agent Telemetry"*,
§5.5 Temporal Behavioral Geometry / §8 Reinforcement Learning Over Latent Behavioral Space

---


## 1. Imports & Path Configuration

In [ ]:
from __future__ import annotations
import sys, warnings, logging
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# -- Simulation core (NB01 baseline)
from src.simulation import (
    TelemetryGenerator, BEHAVIOR_PROFILES,
    PROFILE_NAMES, TELEMETRY_FEATURES,
)

# -- Manifold layer (NB02)
from src.manifold import fit_umap

# -- RL layer (NB05)
from src.rl import (
    BehavioralEnv, make_env_pool,
    QLearningAgent, QLearningConfig,
    ACTION_PUSH_STABLE, ACTION_PUSH_EXPLORATORY, ACTION_DO_NOTHING,
    ACTION_NAMES,
)

# -- Telemetry utilities
from src.telemetry import fit_zscore, apply_zscore

# -- Visualisation
from src.visualization import plot_trajectory_overlay_3d

# -- Data directories (matches NB01-04's convention; note NB05's own DIRS
# cell has a latent bug -- Path("figures/nb05") instead of
# Path("outputs/figures/nb05") -- its actual PNGs on disk were moved there
# by hand after the fact. Not fixed here (out of this notebook's scope) but
# deliberately not repeated below.
DIRS = {
    "raw_nb01"  : Path("data/raw/nb01"),
    "raw_nb02"  : Path("data/raw/nb02"),
    "raw_nb05"  : Path("data/raw/nb05"),
    "raw_nb06"  : Path("data/raw/nb06"),
    "proc_nb06" : Path("data/processed/nb06"),
    "figures"   : Path("outputs/figures/nb06"),
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

def raw(f):     return str(DIRS["raw_nb06"]  / f)
def proc(f):    return str(DIRS["proc_nb06"] / f)
def figfile(f): return str(DIRS["figures"]   / f)
def nb01(f):    return str(DIRS["raw_nb01"]  / f)
def nb02(f):    return str(DIRS["raw_nb02"]  / f)
def nb05(f):    return str(DIRS["raw_nb05"]  / f)

SEED  = 42
FEATS = list(TELEMETRY_FEATURES)
PALETTE = {name: BEHAVIOR_PROFILES[name].color for name in PROFILE_NAMES}

# Phase boundaries match src.rl.adaptation_dynamics.cluster_migration_table's
# convention exactly (early/mid/late as fractions of n_episodes), so phase
# labels here are comparable to NB05's own phase-based figures.
PHASE_BOUNDARIES = (0.33, 0.67)
PHASE_COLORS = {"early": "#e74c3c", "mid": "#f39c12", "late": "#2ecc71"}  # risk -> healthy

# Action colours are motivated by what each action biases the agent toward,
# not arbitrary: push_stable shares stable's green, push_exploratory shares
# exploratory's blue, do_nothing is neutral grey.
ACTION_COLORS = {
    ACTION_NAMES[ACTION_PUSH_STABLE]      : BEHAVIOR_PROFILES["stable"].color,
    ACTION_NAMES[ACTION_PUSH_EXPLORATORY] : BEHAVIOR_PROFILES["exploratory"].color,
    ACTION_NAMES[ACTION_DO_NOTHING]       : "#888888",
}

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white",
    "axes.facecolor": "#fafafa",
    "axes.spines.top": False, "axes.spines.right": False,
})
print("Environment ready.")


## 2. Reconstruct the NB01 Telemetry Pool and the NB02 UMAP Manifold

NB02 saved only the *embedding* (`X_umap3.npy`), not the fitted `umap.UMAP` reducer object,
so there is nothing to call `.transform()` on for new points. We regenerate the identical
20-agent x 2000-step pool (same seed=42) and refit UMAP with NB02's exact hyperparameters
(`configs/projection.yaml`: n_neighbors=30, min_dist=0.10, random_state=42) to get a reducer,
then validate the refit against the saved embedding before trusting it for anything downstream.

In [ ]:
gen = TelemetryGenerator(n_agents=20, n_timesteps=2000, seed=SEED, verbose=False)
gen.run()

X_raw = gen.feature_matrix(z_scored=False)   # (40000, 6) raw units
y     = gen.labels()                          # (40000,) ground-truth regime index

zparams = fit_zscore(X_raw)
X_z     = apply_zscore(X_raw, zparams)

# Sanity check: this should reproduce NB01/NB02's own precomputed z-scored
# columns (gen.feature_matrix(z_scored=True) == data/raw/nb01/X_telemetry.npy)
X_z_precomputed = gen.feature_matrix(z_scored=True)
X_telemetry_saved = np.load(nb01("X_telemetry.npy"))
print("z-score reconstruction check:")
print("  max abs diff (gen internal z-cols vs our fit_zscore) :",
      np.abs(X_z.astype(np.float32) - X_z_precomputed).max())
print("  max abs diff (gen internal z-cols vs NB01 saved .npy):",
      np.abs(X_z_precomputed - X_telemetry_saved).max())


In [ ]:
umap_result = fit_umap(X_z, n_components=3, n_neighbors=30, min_dist=0.10, random_state=SEED)
reducer  = umap_result.reducer
X_umap3_refit = umap_result.embedding

X_umap3_saved = np.load(nb02("X_umap3.npy"))
y_saved       = np.load(nb02("y_labels.npy"))
assert np.array_equal(y, y_saved), "Regenerated labels don't match NB02's saved labels -- pool mismatch."

# Validate the refit against NB02's saved embedding: per-point Euclidean
# distance in the 3-D embedding. UMAP with a fixed random_state on identical
# input data should be highly (though not bit-exactly, depending on
# threading) reproducible within the same library version/environment.
pointwise_dist = np.linalg.norm(X_umap3_refit - X_umap3_saved, axis=1)
print("Refit vs. saved NB02 embedding -- per-point distance (3-D UMAP space):")
print(f"  mean   : {pointwise_dist.mean():.4f}")
print(f"  median : {np.median(pointwise_dist):.4f}")
print(f"  p95    : {np.percentile(pointwise_dist, 95):.4f}")
print(f"  max    : {pointwise_dist.max():.4f}")
print(f"  (for reference, embedding coordinate range is roughly "
      f"{X_umap3_saved.min():.2f} to {X_umap3_saved.max():.2f})")


## 3. Reproduce RL Training With Full Per-Step Trajectory Capture

We retrain 3 agents (indices 0-2, matching NB05's `make_env_pool(base_seed=42)` convention
so seeds line up exactly) with `configs/rl.yaml`'s hyperparameters, this time passing
`collect_trajectories=True` so every step of every episode is retained, not just the last.
The resulting Q-tables are checked against `data/raw/nb05/q_tables.npy` to confirm this
reproduces NB05's actual training result and not merely something similar to it.

In [ ]:
N_TRAJ_AGENTS = 3   # keep this small: 3 x 120 episodes x 500 steps = 180,000 step records

envs = make_env_pool(n_envs=N_TRAJ_AGENTS, base_seed=SEED, delta=0.02, n_steps=500)

rl_cfg_base = dict(alpha=0.15, gamma=0.95, epsilon_start=1.0, epsilon_end=0.05,
                    epsilon_decay=0.97, n_episodes=120)

agents = []
for i, env in enumerate(envs):
    cfg = QLearningConfig(seed=SEED + i, **rl_cfg_base)
    agent = QLearningAgent(env=env, config=cfg)
    agent.train(collect_trajectories=True, verbose=False)
    agents.append(agent)
    print(f"agent {i}: {len(agent.trajectory_log)} episodes captured, "
          f"{sum(len(ep) for ep in agent.trajectory_log)} total step records")


In [ ]:
Q_all_saved = np.load(nb05("q_tables.npy"))  # (20, 100, 3) -- final Q-tables from NB05

print("Retrained vs. NB05-saved final Q-table agreement (first 3 agents):")
for i, agent in enumerate(agents):
    diff = np.abs(agent.Q - Q_all_saved[i])
    print(f"  agent {i}: max abs diff = {diff.max():.6f}, mean abs diff = {diff.mean():.6f}, "
          f"policy agreement = {(agent.Q.argmax(axis=1) == Q_all_saved[i].argmax(axis=1)).mean():.3f}")


## 4. Assemble & Embed the Full Training Trajectory

Flatten each agent's per-episode trajectory list into one long DataFrame, tag every step with
its training phase (early/mid/late, `PHASE_BOUNDARIES` above), z-score the 6 telemetry
features with the *same* `zparams` fit in Section 2, and project into the NB02 manifold via
`reducer.transform()`.

In [ ]:
def flatten_trajectory_log(trajectory_log, agent_idx, phase_boundaries=PHASE_BOUNDARIES):
    n_ep = len(trajectory_log)
    early_end, late_start = int(phase_boundaries[0] * n_ep), int(phase_boundaries[1] * n_ep)
    rows = []
    for ep, ep_traj in enumerate(trajectory_log):
        phase = "early" if ep < early_end else ("late" if ep >= late_start else "mid")
        for rec in ep_traj:
            row = dict(rec)
            row["episode"] = ep
            row["phase"]   = phase
            row["agent_idx"] = agent_idx
            rows.append(row)
    return pd.DataFrame(rows)

traj_dfs = [flatten_trajectory_log(a.trajectory_log, i) for i, a in enumerate(agents)]
traj_df_all = pd.concat(traj_dfs, ignore_index=True)
print(f"Assembled {len(traj_df_all)} step records across {N_TRAJ_AGENTS} agents.")
print(traj_df_all["phase"].value_counts())

X_traj_raw = traj_df_all[FEATS].to_numpy(dtype=np.float64)
X_traj_z   = apply_zscore(X_traj_raw, zparams)
X_traj_umap = reducer.transform(X_traj_z)

traj_df_all["umap1"] = X_traj_umap[:, 0]
traj_df_all["umap2"] = X_traj_umap[:, 1]
traj_df_all["umap3"] = X_traj_umap[:, 2]

out_cols = ["agent_idx", "episode", "episode_step", "phase", "action", "hidden_state",
            "latency", "entropy", "reward", "memory_usage", "error_rate", "action_freq",
            "rl_reward", "umap1", "umap2", "umap3"]
traj_df_all[out_cols].to_csv(proc("rl_trajectory_embedded.csv"), index=False)
print(f"Saved -> {proc('rl_trajectory_embedded.csv')}")


## 5. Overlay Figures -- Training Phase & Action

Both figures use agent 0's full trajectory (60,000 steps, thinned for plotting) over the
complete NB02 background manifold (all 40,000 points, faint, coloured by ground-truth regime).

In [ ]:
THIN = 4  # plot every 4th step of agent 0's trajectory (~15,000 points) for a legible figure
agent0_traj = traj_df_all[traj_df_all["agent_idx"] == 0].iloc[::THIN].reset_index(drop=True)
agent0_umap = agent0_traj[["umap1", "umap2", "umap3"]].to_numpy()

ax = plot_trajectory_overlay_3d(
    background_embedding=X_umap3_saved, background_labels=y_saved,
    overlay_embedding=agent0_umap, overlay_values=agent0_traj["phase"].to_numpy(),
    overlay_kind="categorical", overlay_palette=PHASE_COLORS,
    title="RL Trajectory (agent 0) over NB02 Manifold, by Training Phase",
    overlay_legend_title="training phase",
)
plt.tight_layout()
plt.savefig(figfile("fig_nb06_01_trajectory_by_phase.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Action -1 marks the initial reset observation of each episode (no action was
# taken to reach it) -- excluded here since it isn't the result of any action.
agent0_actions = agent0_traj[agent0_traj["action"] != -1].copy()
agent0_actions["action_name"] = agent0_actions["action"].map(ACTION_NAMES)
agent0_action_umap = agent0_actions[["umap1", "umap2", "umap3"]].to_numpy()

ax = plot_trajectory_overlay_3d(
    background_embedding=X_umap3_saved, background_labels=y_saved,
    overlay_embedding=agent0_action_umap, overlay_values=agent0_actions["action_name"].to_numpy(),
    overlay_kind="categorical", overlay_palette=ACTION_COLORS,
    title="RL Trajectory (agent 0) over NB02 Manifold, by Action Taken",
    overlay_legend_title="action",
)
plt.tight_layout()
plt.savefig(figfile("fig_nb06_02_trajectory_by_action.png"), dpi=150, bbox_inches="tight")
plt.show()


## 6. Early vs. Late Training Comparison

If training is actually reshaping where the agent spends time, the early-phase footprint
should sit closer to the unstable region than the late-phase footprint. Quantified here as
mean distance (in UMAP space) from each phase's points to the unstable and stable regime
centroids, alongside the same comparison as a figure.

In [ ]:
centroid_unstable = X_umap3_saved[y_saved == PROFILE_NAMES.index("unstable")].mean(axis=0)
centroid_stable   = X_umap3_saved[y_saved == PROFILE_NAMES.index("stable")].mean(axis=0)

def dist_to(points, centroid):
    return np.linalg.norm(points - centroid, axis=1)

rows = []
for agent_idx in range(N_TRAJ_AGENTS):
    sub = traj_df_all[traj_df_all["agent_idx"] == agent_idx]
    for phase in ["early", "mid", "late"]:
        p = sub[sub["phase"] == phase]
        pts = p[["umap1", "umap2", "umap3"]].to_numpy()
        rows.append({
            "agent_idx": agent_idx, "phase": phase, "n_steps": len(p),
            "mean_dist_unstable_centroid": float(dist_to(pts, centroid_unstable).mean()),
            "mean_dist_stable_centroid"  : float(dist_to(pts, centroid_stable).mean()),
        })
phase_dist_df = pd.DataFrame(rows)
phase_dist_df.to_csv(proc("phase_manifold_distance_summary.csv"), index=False)
phase_dist_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), subplot_kw={"projection": "3d"})

for ax, phase, title in zip(axes, ["early", "late"], ["Early Training (episodes 0-39)", "Late Training (episodes 80-119)"]):
    sub = agent0_traj if phase in agent0_traj["phase"].values else traj_df_all[traj_df_all["agent_idx"] == 0]
    sub = traj_df_all[(traj_df_all["agent_idx"] == 0) & (traj_df_all["phase"] == phase)].iloc[::2]
    pts = sub[["umap1", "umap2", "umap3"]].to_numpy()
    plot_trajectory_overlay_3d(
        background_embedding=X_umap3_saved, background_labels=y_saved,
        overlay_embedding=pts, overlay_values=np.full(len(pts), phase),
        overlay_kind="categorical", overlay_palette=PHASE_COLORS,
        ax=ax, title=title, overlay_legend_title="phase",
    )

plt.tight_layout()
plt.savefig(figfile("fig_nb06_03_early_vs_late.png"), dpi=150, bbox_inches="tight")
plt.show()


## 7. Multi-Agent Robustness Check

Repeats the phase-coloured overlay for agents 1 and 2, to check whether the pattern seen for
agent 0 is a general effect of training or an artifact of one agent's particular random seed.

In [ ]:
fig, axes = plt.subplots(1, N_TRAJ_AGENTS, figsize=(7 * N_TRAJ_AGENTS, 7), subplot_kw={"projection": "3d"})

for ax, agent_idx in zip(axes, range(N_TRAJ_AGENTS)):
    sub = traj_df_all[traj_df_all["agent_idx"] == agent_idx].iloc[::THIN]
    pts = sub[["umap1", "umap2", "umap3"]].to_numpy()
    plot_trajectory_overlay_3d(
        background_embedding=X_umap3_saved, background_labels=y_saved,
        overlay_embedding=pts, overlay_values=sub["phase"].to_numpy(),
        overlay_kind="categorical", overlay_palette=PHASE_COLORS,
        ax=ax, title=f"agent {agent_idx}", overlay_legend_title="phase",
        background_s=1.5, overlay_s=6,
    )

plt.tight_layout()
plt.savefig(figfile("fig_nb06_04_multi_agent_phase.png"), dpi=150, bbox_inches="tight")
plt.show()


## 8. Summary & Findings

In [ ]:
print("NB06 Findings")
print("=" * 70)
umap_match = "close" if pointwise_dist.mean() < 1.0 else "DIVERGED -- check UMAP determinism"
print(f"UMAP refit vs. NB02 saved embedding : {umap_match} (mean dist={pointwise_dist.mean():.4f})")
q_diffs = [float(np.abs(a.Q - Q_all_saved[i]).max()) for i, a in enumerate(agents)]
print(f"Retrained Q-tables vs. NB05 saved   : max abs diff per agent = {[round(d,4) for d in q_diffs]}")
print()
print(phase_dist_df.groupby("phase")[["mean_dist_unstable_centroid", "mean_dist_stable_centroid"]].mean())


### Notebook 06 Findings

| Finding | Evidence |
|---------|----------|
| **UMAP reducer reproduces NB02's saved embedding** | see refit-vs-saved distance check, Section 2 |
| **Retraining reproduces NB05's actual result** | Q-table diff vs. `q_tables.npy`, Section 3 |
| **Training-phase separation in manifold space** | early/mid/late mean distance to unstable/stable centroids, Section 6 |
| **Pattern holds across agents** | multi-agent overlay, Section 7 |
| **Reward-key bug found and fixed in `src/rl/environment.py`** | `.trajectory` records were silently overwriting the telemetry `"reward"` feature with the RL step's shaped reward on every step; renamed to `"rl_reward"`. Q-learning itself was unaffected (it reads `StepResult.info`, not `.trajectory`) -- see `environment.py` step()/reset() docstrings. |

*(Interpretive prose for the numeric findings above intentionally left for a human pass after
reviewing the executed outputs and figures -- the printed/saved numbers in Sections 2, 3, and 6
are the evidence; this table should be filled in against the actual run, not assumed.)*
